# <span style="color:red; font-size:30px"> Lecture 20c: Cleaning Data </span>

## <span style="color:red"> I. Import Libraries and Read in Files</span>

In [1]:
import numpy as np
import pandas as pd

circuits = pd.read_csv("f1_data/circuits.csv")

# <span style="color:red"> II. Examine Data </span>

In [11]:
# column/variable headings
print(circuits.columns.values)
print()
# number of rows and columns of DataFrame
print(circuits.shape)
print()
n_rows, n_cols = circuits.shape
print("# of rows:", n_rows)
print("# of columns:", n_cols)


['circuitId' 'circuitRef' 'name' 'location' 'country' 'lat' 'lng' 'alt'
 'url' 'alt_numeric']

(76, 10)

# of rows: 76
# of columns: 10


<font size = "4">

We can also get the **DataTypes** for each column (dtypes)

In [ ]:
print(circuits.dtypes)

<font size = "4">

We've seen **attributes** (a.k.a properties) of Pandas DataFrames like ".shape", ".dtypes", and ".columns"

We've also seen **methods** like ".query", ".apply", and ".mean"


How do we remember all of these?

VS code can remind you of all available methods for an object.

If you want to see attributes as well, you should do the following.

1. Install the Pylance extension in your VS Code

2. Click the box at the center-top of your VS Code window

3. Start typing "> Developer: Reload Window". It will autocomplete after you type a sufficient number of the characters, then hit enter.

4. This will also restart the notebook, so you should re-run the cells above

In the code cell below, type "circuits." (with the period) and then wait, without running the cell.

In [ ]:
# Type the following (including the period), then wait, without running the cell
# type this (with the period) >> circuits.
circuits.


<font size = "4">

**Note:** We've already seen how to add new columns/variables to a DataFrame. But we haven't seen how to rename the existing columns/variables. Some of the most straightforward are shown below

In [ ]:
# Change the first and last column headings
print(circuits.columns.values)
print()

# Change last heading to "Uniform Resource Locator"
circuits.columns.values[-1] = "Uniform Resource Locator"

# Change first heading to "ID_No"
circuits.columns.values[0] = "ID_No"
print(circuits.columns.values)

In [ ]:
# Change back to "url" 

# Create a new data-frame and assign it to "circuits". 
# So we're overwriting the dataframe.
circuits = circuits.rename(columns={'Uniform Resource Locator': 'url'})

print(circuits.columns.values)

In [ ]:
# Change back to "circuitID"

# Instead of making a new dataframe and over-writing the old one, change it "in-place"
circuits.rename(columns={'ID_No':'circuitID'}, inplace=True)

print(circuits.columns.values)

# <span style="color:red"> III. Cleaning Data </span>

<font size = "4">

Data that should be numerical (e.g. altitude of a Formula One track) can have missing values or non-numerical replacements (like "N/A"). We want to manipulate the data to allow for numerical calculations.

In [ ]:
print(circuits["lat"].mean())
print(circuits["lng"].mean())
print(circuits["alt"].mean())

In [ ]:
# Let's try to figure out the problem
print(circuits.dtypes)

# The column "alt" has the DataType "object"

In [ ]:
print(circuits["alt"])
# these look like numbers, except the last two are "\N"

In [ ]:
display(circuits["alt"])

<font size = "4">

- Each column of a Pandas DataFrame is a Pandas Series
- Each Pandas Series with "dtype: object" has a collection of methods known as "String Methods"
- To execute a method from this collection you must type: ``series_name.str.method_name``
- **NOTE**: Whenever I type something with a vague placeholder name like "series_name" or "method_name" this means that you should replace that word/phrase with an actual Python object or method/function name
- Within this collection, there is a method we want to use called ``.isnumeric``

In [ ]:
# This is a Pandas Series
print(type(circuits["alt"])) 
print()
# This is a Pandas StringMethods object (collection of methods/functions)
print(type(circuits["alt"].str)) 
print()
# This is a method/function belonging to the StringMethods collection
print(type(circuits["alt"].str.isnumeric)) 

<font size = "4">

You'll get an error if you try to use ``.str`` with a column that doesn't have the "object" dtype

In [ ]:
circuits["lat"].str

<font size = "4">

The ``series_name.str.isnumeric()`` method returns a Pandas Series with ``True`` everywhere the Series entry is deemed "numeric" and ``False`` otherwise.

In [2]:
print(circuits["alt"].str.isnumeric())

0      True
1      True
2      True
3      True
4      True
      ...  
71     True
72     True
73     True
74    False
75    False
Name: alt, Length: 76, dtype: bool


In [3]:
# We can reference subattributes of columns in ".query()"
non_numeric = circuits.query("  alt.str.isnumeric() == False ")
print(non_numeric)

    circuitId circuitRef                           name   location  \
70         73       baku              Baku City Circuit       Baku   
74         78     losail   Losail International Circuit  Al Daayen   
75         79      miami  Miami International Autodrome      Miami   

       country      lat      lng alt  \
70  Azerbaijan  40.3725  49.8533  -7   
74       Qatar  25.4900  51.4542  \N   
75         USA  25.9581 -80.2389  \N   

                                                  url  
70     http://en.wikipedia.org/wiki/Baku_City_Circuit  
74  http://en.wikipedia.org/wiki/Losail_Internatio...  
75  http://en.wikipedia.org/wiki/Miami_Internation...  


<font size = "4">

Why is the string "-7" considered non-numeric?

In [4]:
# The pd.unique() function extracts unique values from a series/array

unique_non_numeric = pd.unique( non_numeric["alt"] )
print(unique_non_numeric)

['-7' '\\N']


<font size = "4">

We'll decide what we should replace "\N" and "-7" with

- "-7" is easy. We'll replace the string "-7" with the integer -7
- What about "\N"? It turns out the best choice is the "nan" (not a number) object from the Numpy library.

In [5]:
print(np.nan)

# ironically, the type of this object is a floating-point number...
print(type(np.nan))

nan
<class 'float'>


In [6]:
# unique_non_numeric is the list with ["-7", "\\N"]
# we'll make a list of the same length with the values we want to replace them with

replace_vals = [-7, np.nan]

# Overwrite the "alt" column, replacing every appearance from unique_non_numeric with
# the corresponding element of replace_vals
circuits["alt"] = circuits["alt"].replace(unique_non_numeric, replace_vals)

<font size = "4">

Did it work? Let's test it

In [7]:
non_numeric = circuits.query("  alt.str.isnumeric() == False  ")
print(non_numeric)

Empty DataFrame
Columns: [circuitId, circuitRef, name, location, country, lat, lng, alt, url]
Index: []


<font size = "4">

There are no rows with a non-numeric value for "alt"! Are we done? (No)

In [8]:
print(circuits["alt"].mean()) # get an error

TypeError: can only concatenate str (not "int") to str

<font size = "4">

What happened? We still don't have the right "dtype" to do calculations. We should have checked that in the first place

In [ ]:
print(circuits.dtypes)

<font size = "4">

The "alt" column still has the "object" dtype. But since we've cleaned the data, all of its elements **can be** converted to numeric values. The ``Pandas`` function ``to_numeric`` can be used to do this.

We'll create a new column to the DataFrame.

In [9]:
circuits["alt_numeric"] = pd.to_numeric(circuits["alt"])
print(circuits["alt_numeric"].mean())

248.1891891891892


In [10]:
# After the cleaning process is done, you may want to store the dataset again
# It's recommended to do this in a separate file from the original
# That way you can always go back to the original if you made a programming error

circuits.to_csv("f1_clean_data/circuits.csv")

<font size = "4">

- By the way, for this example, we can clean the altitude data with one line.

- We'll reload the data and show how.

In [ ]:
circuits_2 = pd.read_csv("f1_data/circuits.csv")

circuits_2["alt_numeric"] = pd.to_numeric(circuits_2["alt"], errors="coerce")

print(circuits_2["alt_numeric"].mean())

<font size = "4">

**Warning:** We cannot just "force-clean" every dataset like this. See the last exercise for an example where you CAN'T do this.

<font size = "4">

**Exercise**: When I was creating this notebook, I had the following question. "If I read in the .csv file I just created and save it as a new DataFrame, will I have to re-do any part of the cleaning process?" Test this out for yourself.

**Exercise**: Read in the .csv file that was just created, saving it as a new DataFrame. You'll see there is an extra column/variable called "Unnamed" with the values 0, 1, 2, ..., 75, 76. Type ``help(circuits.to_csv)`` to figure out how to create a new .csv file without this extra column.

In [ ]:
# your answer here

In [ ]:
# your answer here


<font size = "4">

**Exercise:**

- Use ".replace()" with the "country" column
- Replace "USA" with "United States"

In [ ]:
# Write your own code





<font size = "4">

**Exercise:** 

Below, I manually create a DataFrame using made-up survey data. Each respondent is a student, and their answers were read in using Python code as strings.

- Clean the "age" column so that we can calculate the mean age of the respondents. Will using `pd.to_numeric` with `errors="coerce"` clean the data correctly?

<br>

<font size = "3">

Note: The line with curly braces ``{}`` creates a *dictionary* object, which is then converted to a Pandas DataFrame. We also saw this earlier in the notebook within the `.replace` method. We'll discuss dictionaries more in the next lecture.

In [ ]:
list_ages = ["twenty-one", "21", "18", "twenty", "", "21", "twenty five", "18", "twenty", "19", 
    "20", "nineteen", "", "17"]
list_major = ["DSci", "Math", "undeclared", "Undeclared", "Sociology", "psychology", "Spanish", 
    "Ancient Mediterranean Studies", "Bio", "biology", "Bus Admin.", "BBA", "Linguistics", ""]
list_graduate = ["Y", "Y", "N", "Y", "N", "N", "N", "N", "Y", "N", "N", "N", "Y", "N"]

data_dictionary = {"age": list_ages, "major": list_major, "graduating?": list_graduate}

df_survey = pd.DataFrame(data_dictionary)

display(df_survey)